# 🏠 Home Guardian — YOLOv8 Door & Window State Detector
### Production Training Notebook | Maximum Precision Configuration
---

## ⚠️ STEP 0 — Prevent Colab Session Disconnect

Training takes a long time. Colab will disconnect your session if it detects no interaction.
**Before starting training**, do the following:

1. In Chrome/Edge, press **F12** to open Developer Tools
2. Click the **Console** tab
3. Paste the code below and press **Enter**

```javascript
// Colab keep-alive — clicks the page every 60 seconds
function ColabKeepAlive() {
  document.querySelector('colab-toolbar-button#connect').click();
  console.log('Keep-alive ping sent at: ' + new Date().toLocaleTimeString());
}
const keepAliveInterval = setInterval(ColabKeepAlive, 60000);
console.log('Keep-alive started. To stop it, run: clearInterval(keepAliveInterval)');
```

4. You should see **"Keep-alive started"** in the console
5. To stop it later run: `clearInterval(keepAliveInterval)`

---
> **Note:** Also go to **Edit → Notebook settings → Hardware accelerator → GPU (T4)**
> before running any cell.

## 📁 STEP 1 — Configuration
Set your project path here. This is the only cell you need to edit.

In [ ]:
# ============================================================
# PROJECT CONFIGURATION — Edit these values only
# ============================================================

# Full path to your project folder inside Google Drive
# Example: '/content/drive/MyDrive/HomeGuardian/YOLOv8'
PROJECT_PATH = '/content/drive/MyDrive/HomeGuardian/YOLOv8'

# Path to your data.yaml file (relative to PROJECT_PATH)
DATA_YAML = 'data.yaml'

# YOLOv8 model size: 'yolov8n.pt' (fast) | 'yolov8s.pt' (balanced) | 'yolov8m.pt' (accurate)
BASE_MODEL = 'yolov8s.pt'

# Training run name (saved inside PROJECT_PATH/runs/)
RUN_NAME = 'homeguardian_v1'

# Inference test — path to a test image or video after training
# Example: 'test_samples/living_room.jpg'
TEST_MEDIA_PATH = 'test_samples/test.jpg'

# ============================================================
print('Configuration loaded.')
print(f'  Project path : {PROJECT_PATH}')
print(f'  Data YAML    : {DATA_YAML}')
print(f'  Base model   : {BASE_MODEL}')
print(f'  Run name     : {RUN_NAME}')

## ☁️ STEP 2 — Mount Google Drive & Set Working Directory

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)
print('Google Drive mounted successfully.')

# Validate the project path exists
if not os.path.exists(PROJECT_PATH):
    raise FileNotFoundError(
        f'Project path not found: {PROJECT_PATH}\n'
        f'Please create the folder in Drive and upload your dataset there.'
    )

# Change working directory to project folder
os.chdir(PROJECT_PATH)
print(f'Working directory set to: {os.getcwd()}')

# Validate data.yaml exists
if not os.path.exists(DATA_YAML):
    raise FileNotFoundError(
        f'data.yaml not found at: {os.path.join(PROJECT_PATH, DATA_YAML)}\n'
        f'Export your Roboflow dataset and place data.yaml in the project folder.'
    )

print(f'data.yaml found.')

# List contents so you can visually confirm structure
print('\nProject folder contents:')
for item in sorted(os.listdir('.')):
    print(f'  {item}')

## 🖥️ STEP 3 — GPU Check & Environment Setup

In [ ]:
import subprocess

# Check GPU availability and print specs
print('=' * 55)
print('GPU ENVIRONMENT CHECK')
print('=' * 55)

gpu_info = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
     '--format=csv,noheader'],
    capture_output=True, text=True
)

if gpu_info.returncode == 0:
    gpu_line = gpu_info.stdout.strip()
    parts = gpu_line.split(',')
    print(f'  GPU Name      : {parts[0].strip()}')
    print(f'  VRAM          : {parts[1].strip()}')
    print(f'  Driver        : {parts[2].strip()}')
else:
    print('  WARNING: No GPU detected!')
    print('  Go to Runtime > Change runtime type > GPU')
    raise EnvironmentError('GPU required for training. No GPU found.')

print('=' * 55)

In [ ]:
# Install ultralytics quietly
subprocess.run(['pip', 'install', 'ultralytics', '-q'], check=True)
print('ultralytics installed.')

# Core imports
import torch
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from IPython.display import display, Image as IPImage
from ultralytics import YOLO

# Confirm torch sees the GPU
print(f'PyTorch version  : {torch.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device      : {torch.cuda.get_device_name(0)}')
    print(f'CUDA version     : {torch.version.cuda}')

# Set plot style globally
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('\nAll imports successful. Environment is ready.')

## 🧱 STEP 4 — Background Image Setup (Anti-False-Positive)

Background images are images with **no objects** — empty rooms, walls, floors.
YOLO uses them to learn what is NOT a door or window, directly reducing false positives.

Place your background images in a folder called `background/` inside your project.
This cell creates empty label files for each one (required by YOLO format).

In [ ]:
import glob
import shutil

BACKGROUND_IMAGES_DIR = os.path.join(PROJECT_PATH, 'background')
BACKGROUND_LABELS_DIR = os.path.join(PROJECT_PATH, 'background_labels')

if os.path.exists(BACKGROUND_IMAGES_DIR):
    os.makedirs(BACKGROUND_LABELS_DIR, exist_ok=True)

    bg_images = glob.glob(os.path.join(BACKGROUND_IMAGES_DIR, '*.jpg')) + \
                glob.glob(os.path.join(BACKGROUND_IMAGES_DIR, '*.png')) + \
                glob.glob(os.path.join(BACKGROUND_IMAGES_DIR, '*.jpeg'))

    created = 0
    for img_path in bg_images:
        label_name = Path(img_path).stem + '.txt'
        label_path = os.path.join(BACKGROUND_LABELS_DIR, label_name)
        # Empty .txt file = no objects = background image
        with open(label_path, 'w') as f:
            pass
        created += 1

    print(f'Background setup complete.')
    print(f'  Images found  : {len(bg_images)}')
    print(f'  Labels created: {created}')
    print(f'\nYOLO will use these to learn what is NOT a door/window.')
    print(f'Add these paths to your data.yaml train list if not already included.')
else:
    print('No background/ folder found — skipping background image setup.')
    print('TIP: Create a background/ folder and add 50-100 empty room images')
    print('     for significantly better false positive suppression.')

## 🔍 STEP 5 — Dataset Validation
Before training, verify the dataset structure and class distribution.

In [ ]:
import yaml

# Load and display data.yaml
with open(DATA_YAML, 'r') as f:
    data_config = yaml.safe_load(f)

print('=' * 55)
print('DATASET CONFIGURATION')
print('=' * 55)
print(f'  Classes ({data_config["nc"]}) : {data_config["names"]}')
print(f'  Train path : {data_config.get("train", "not set")}')
print(f'  Val path   : {data_config.get("val", "not set")}')
print(f'  Test path  : {data_config.get("test", "not set")}')
print('=' * 55)

# Count images and labels per split
def count_split(split_path):
    if not split_path or not os.path.exists(split_path):
        return 0, 0
    # Handle both images/ subfolder and direct path
    img_dir = split_path if os.path.isdir(split_path) else os.path.dirname(split_path)
    images = glob.glob(os.path.join(img_dir, '*.jpg')) + \
             glob.glob(os.path.join(img_dir, '*.png')) + \
             glob.glob(os.path.join(img_dir, '*.jpeg'))
    label_dir = img_dir.replace('images', 'labels')
    labels = glob.glob(os.path.join(label_dir, '*.txt')) if os.path.exists(label_dir) else []
    return len(images), len(labels)

for split in ['train', 'val', 'test']:
    split_path = data_config.get(split, '')
    if split_path:
        full_path = split_path if os.path.isabs(split_path) else os.path.join(PROJECT_PATH, split_path)
        imgs, lbls = count_split(full_path)
        print(f'  {split:5s} → {imgs:4d} images | {lbls:4d} label files')

# Count annotations per class
print('\nAnnotation count per class:')
class_names = data_config['names']
class_counts = {name: 0 for name in class_names}

train_path = data_config.get('train', '')
if train_path:
    full_train = train_path if os.path.isabs(train_path) else os.path.join(PROJECT_PATH, train_path)
    label_dir = full_train.replace('images', 'labels')
    if os.path.exists(label_dir):
        for lf in glob.glob(os.path.join(label_dir, '*.txt')):
            with open(lf, 'r') as f:
                for line in f:
                    line = line.strip()
                    if line:
                        class_id = int(line.split()[0])
                        if class_id < len(class_names):
                            class_counts[class_names[class_id]] += 1

for cls, count in class_counts.items():
    bar = '█' * (count // 50)
    print(f'  {cls:20s} : {count:5d}  {bar}')

print('\nDataset validation complete.')

## 🚀 STEP 6 — Training with Maximum Precision Hyperparameters

In [ ]:
# Load base model
model = YOLO(BASE_MODEL)
print(f'Loaded base model: {BASE_MODEL}')
print(f'Starting training run: {RUN_NAME}')
print('=' * 55)

results = model.train(
    # ── Core Setup ──────────────────────────────────────────
    data=DATA_YAML,
    project=os.path.join(PROJECT_PATH, 'runs'),
    name=RUN_NAME,
    exist_ok=True,

    # ── Image & Batch ────────────────────────────────────────
    imgsz=640,
    batch=16,              # reduce to 8 if you get CUDA out-of-memory
    workers=4,

    # ── Training Duration ────────────────────────────────────
    epochs=100,
    patience=15,           # early stopping: stops if no improvement for 15 epochs

    # ── Optimizer ───────────────────────────────────────────
    optimizer='AdamW',     # better convergence than SGD for this task
    lr0=0.001,             # initial learning rate
    lrf=0.01,              # final lr = lr0 * lrf
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,       # gradual warmup prevents early instability
    warmup_momentum=0.8,

    # ── Loss Weights ────────────────────────────────────────
    # Increasing cls loss forces model to be more selective
    # before committing to a class — reduces false positives
    box=7.5,               # bounding box regression loss weight
    cls=1.5,               # classification loss weight (higher = more selective)
    dfl=1.5,               # distribution focal loss weight

    # ── Augmentation ────────────────────────────────────────
    hsv_h=0.015,           # hue variation (slight)
    hsv_s=0.5,             # saturation variation (moderate)
    hsv_v=0.4,             # brightness variation (handles day/night)
    degrees=5.0,           # slight rotation (cameras are never perfectly level)
    translate=0.1,         # slight position shift
    scale=0.5,             # zoom variation
    shear=0.0,             # no shear (windows/doors are always rectangular)
    perspective=0.0001,    # very slight perspective (camera angle changes)
    flipud=0.0,            # no vertical flip (upside-down doors don't exist)
    fliplr=0.5,            # horizontal flip (left/right symmetric)
    mosaic=1.0,            # mosaic ON during most of training
    mixup=0.1,             # slight mixup augmentation
    copy_paste=0.0,        # off — not needed for static scenes
    close_mosaic=10,       # turn off mosaic in final 10 epochs for clean bbox alignment

    # ── Precision & Output ───────────────────────────────────
    conf=0.001,            # low conf during training to capture all predictions for mAP
    iou=0.7,               # IoU threshold for NMS — higher = stricter duplicate suppression
    max_det=50,            # max detections per image (a room has at most a few doors/windows)
    plots=True,            # save built-in training plots
    save=True,
    save_period=10,        # checkpoint every 10 epochs
    val=True,
    verbose=True,
    device=0,              # GPU 0
    seed=42,               # reproducibility
)

print('\n' + '=' * 55)
print('TRAINING COMPLETE')
print('=' * 55)
BEST_MODEL_PATH = os.path.join(PROJECT_PATH, 'runs', RUN_NAME, 'weights', 'best.pt')
print(f'Best model saved at: {BEST_MODEL_PATH}')

## 📊 STEP 7 — Advanced Training Curves & Metrics

In [ ]:
# Load results.csv
results_csv = os.path.join(PROJECT_PATH, 'runs', RUN_NAME, 'results.csv')

if not os.path.exists(results_csv):
    raise FileNotFoundError(f'results.csv not found at {results_csv}')

df = pd.read_csv(results_csv)

# Strip whitespace from column names — prevents NameError on YOLO output
df.columns = df.columns.str.strip()

print('Columns found in results.csv:')
print(list(df.columns))
print(f'\nTotal epochs trained: {len(df)}')
print(f'Best mAP50         : {df["metrics/mAP50(B)"].max():.4f} at epoch {df["metrics/mAP50(B)"].idxmax() + 1}')
print(f'Best mAP50-95      : {df["metrics/mAP50-95(B)"].max():.4f} at epoch {df["metrics/mAP50-95(B)"].idxmax() + 1}')
print(f'Best Precision     : {df["metrics/precision(B)"].max():.4f}')
print(f'Best Recall        : {df["metrics/recall(B)"].max():.4f}')

In [ ]:
epochs = df['epoch'] if 'epoch' in df.columns else range(1, len(df) + 1)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Home Guardian — YOLOv8 Training Results', fontsize=16, fontweight='bold', y=1.01)

# ── Plot 1: mAP Curves ──────────────────────────────────────
ax = axes[0, 0]
ax.plot(epochs, df['metrics/mAP50(B)'], color='#2196F3', linewidth=2, label='mAP50')
ax.plot(epochs, df['metrics/mAP50-95(B)'], color='#9C27B0', linewidth=2, label='mAP50-95')
best_epoch = df['metrics/mAP50(B)'].idxmax()
ax.axvline(x=best_epoch + 1, color='red', linestyle='--', alpha=0.5, label=f'Best epoch ({best_epoch+1})')
ax.set_title('mAP Curves', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('mAP')
ax.legend()
ax.set_ylim(0, 1)

# ── Plot 2: Precision & Recall ───────────────────────────────
ax = axes[0, 1]
ax.plot(epochs, df['metrics/precision(B)'], color='#4CAF50', linewidth=2, label='Precision')
ax.plot(epochs, df['metrics/recall(B)'], color='#FF9800', linewidth=2, label='Recall')
ax.set_title('Precision & Recall', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Score')
ax.legend()
ax.set_ylim(0, 1)

# ── Plot 3: Loss Curves ──────────────────────────────────────
ax = axes[0, 2]
if 'train/box_loss' in df.columns:
    ax.plot(epochs, df['train/box_loss'], color='#F44336', linewidth=2, label='Box loss')
if 'train/cls_loss' in df.columns:
    ax.plot(epochs, df['train/cls_loss'], color='#2196F3', linewidth=2, label='Cls loss')
if 'train/dfl_loss' in df.columns:
    ax.plot(epochs, df['train/dfl_loss'], color='#4CAF50', linewidth=2, label='DFL loss')
ax.set_title('Training Losses', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()

# ── Plot 4: Val Loss Curves ──────────────────────────────────
ax = axes[1, 0]
if 'val/box_loss' in df.columns:
    ax.plot(epochs, df['val/box_loss'], color='#F44336', linewidth=2, label='Val box loss')
if 'val/cls_loss' in df.columns:
    ax.plot(epochs, df['val/cls_loss'], color='#2196F3', linewidth=2, label='Val cls loss')
if 'val/dfl_loss' in df.columns:
    ax.plot(epochs, df['val/dfl_loss'], color='#4CAF50', linewidth=2, label='Val DFL loss')
ax.set_title('Validation Losses', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()

# ── Plot 5: Learning Rate Schedule ──────────────────────────
ax = axes[1, 1]
if 'lr/pg0' in df.columns:
    ax.plot(epochs, df['lr/pg0'], color='#607D8B', linewidth=2, label='LR (group 0)')
if 'lr/pg1' in df.columns:
    ax.plot(epochs, df['lr/pg1'], color='#795548', linewidth=2, linestyle='--', label='LR (group 1)')
ax.set_title('Learning Rate Schedule', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.legend()

# ── Plot 6: F1 Score (computed from P and R) ─────────────────
ax = axes[1, 2]
precision = df['metrics/precision(B)']
recall = df['metrics/recall(B)']
f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
ax.plot(epochs, f1, color='#E91E63', linewidth=2, label='F1 Score')
best_f1_epoch = f1.idxmax()
ax.axvline(x=best_f1_epoch + 1, color='gray', linestyle='--', alpha=0.5,
           label=f'Best F1={f1.max():.3f} (ep {best_f1_epoch+1})')
ax.set_title('F1 Score Over Epochs', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('F1')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plot_save_path = os.path.join(PROJECT_PATH, 'runs', RUN_NAME, 'custom_training_curves.png')
plt.savefig(plot_save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved to: {plot_save_path}')

## ✅ STEP 8 — Validation: Per-Class Precision Metrics

In [ ]:
# Load best model and run full validation
best_model = YOLO(BEST_MODEL_PATH)
print(f'Running validation with best model: {BEST_MODEL_PATH}')
print('=' * 55)

val_results = best_model.val(
    data=DATA_YAML,
    imgsz=640,
    conf=0.50,
    iou=0.6,
    device=0,
    plots=True,
    save_json=False,
    verbose=True
)

# Extract and display per-class metrics cleanly
print('\n' + '=' * 55)
print('PER-CLASS VALIDATION RESULTS')
print('=' * 55)

class_names_list = list(data_config['names']) if isinstance(data_config['names'], dict) else data_config['names']

try:
    maps = val_results.maps          # mAP per class array
    p_vals = val_results.box.p       # precision per class
    r_vals = val_results.box.r       # recall per class
    ap50 = val_results.box.ap50      # AP@50 per class
    ap = val_results.box.ap          # AP@50-95 per class

    print(f'  {"Class":<22} {"Precision":>10} {"Recall":>10} {"AP50":>10} {"AP50-95":>10}')
    print(f'  {"-"*22} {"-"*10} {"-"*10} {"-"*10} {"-"*10}')

    for i, cls_name in enumerate(class_names_list):
        if i < len(p_vals):
            p = p_vals[i] if hasattr(p_vals, '__len__') else p_vals
            r = r_vals[i] if hasattr(r_vals, '__len__') else r_vals
            a50 = ap50[i] if hasattr(ap50, '__len__') else ap50
            a = ap[i] if hasattr(ap, '__len__') else ap
            flag = ' ⚠️ LOW' if a50 < 0.75 else ' ✅'
            print(f'  {cls_name:<22} {p:>10.4f} {r:>10.4f} {a50:>10.4f} {a:>10.4f}{flag}')

    print(f'  {"─"*64}')
    print(f'  {"Overall mAP50":<22} {"":>10} {"":>10} {val_results.box.map50:>10.4f}')
    print(f'  {"Overall mAP50-95":<22} {"":>10} {"":>10} {"":>10} {val_results.box.map:>10.4f}')

except AttributeError as e:
    print(f'Note: Could not extract per-class metrics via API ({e})')
    print('Check the runs/ folder for per_class_results.csv generated by YOLO.')

print('=' * 55)

## 🔬 STEP 9 — Confusion Matrix Visualization

In [ ]:
# Display the confusion matrix saved by YOLO during validation
confusion_matrix_path = os.path.join(PROJECT_PATH, 'runs', RUN_NAME, 'confusion_matrix_normalized.png')

if os.path.exists(confusion_matrix_path):
    print('Confusion Matrix (Normalized):')
    display(IPImage(confusion_matrix_path, width=700))
else:
    # Try non-normalized version
    alt_path = os.path.join(PROJECT_PATH, 'runs', RUN_NAME, 'confusion_matrix.png')
    if os.path.exists(alt_path):
        print('Confusion Matrix:')
        display(IPImage(alt_path, width=700))
    else:
        print('Confusion matrix image not found.')
        print(f'Check: {os.path.join(PROJECT_PATH, "runs", RUN_NAME)}')

print('\nHow to read this:')
print('  Diagonal = correct predictions (want these HIGH)')
print('  Off-diagonal = misclassifications (want these LOW)')
print('  Background row = false positives (want this LOW)')

## 🎯 STEP 10 — Edge Inference Test on Real Footage

In [ ]:
import time

TEST_MEDIA_FULL = os.path.join(PROJECT_PATH, TEST_MEDIA_PATH)

if not os.path.exists(TEST_MEDIA_FULL):
    print(f'Test media not found at: {TEST_MEDIA_FULL}')
    print('Update TEST_MEDIA_PATH in STEP 1 to point to a test image or video.')
else:
    print(f'Running inference on: {TEST_MEDIA_FULL}')
    print(f'Confidence threshold: 0.50')
    print('=' * 55)

    start_time = time.time()

    inference_results = best_model.predict(
        source=TEST_MEDIA_FULL,
        conf=0.50,          # production threshold — tune this after reviewing results
        iou=0.6,
        imgsz=640,
        device=0,
        max_det=50,
        save=True,
        save_txt=False,
        show_conf=True,
        show_labels=True,
        project=os.path.join(PROJECT_PATH, 'runs'),
        name=f'{RUN_NAME}_inference',
        exist_ok=True,
        verbose=False
    )

    elapsed = time.time() - start_time

    # Print detection summary
    print(f'Inference completed in {elapsed:.2f}s')
    print(f'Frames/images processed: {len(inference_results)}')

    total_detections = 0
    class_detection_counts = {name: 0 for name in class_names_list}

    for r in inference_results:
        if r.boxes is not None and len(r.boxes) > 0:
            for box in r.boxes:
                cls_id = int(box.cls[0].item())
                conf_val = float(box.conf[0].item())
                cls_name = class_names_list[cls_id] if cls_id < len(class_names_list) else str(cls_id)
                class_detection_counts[cls_name] += 1
                total_detections += 1

    print(f'\nDetection Summary:')
    print(f'  Total detections : {total_detections}')
    for cls_name, count in class_detection_counts.items():
        print(f'  {cls_name:<22}: {count}')

    # Display annotated output if it's an image
    is_image = TEST_MEDIA_FULL.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
    if is_image:
        output_dir = os.path.join(PROJECT_PATH, 'runs', f'{RUN_NAME}_inference')
        output_filename = os.path.basename(TEST_MEDIA_FULL)
        output_path = os.path.join(output_dir, output_filename)

        if os.path.exists(output_path):
            print(f'\nAnnotated image saved to: {output_path}')

            # Display inline using matplotlib for better control
            img = cv2.imread(output_path)
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            fig, ax = plt.subplots(1, 1, figsize=(12, 8))
            ax.imshow(img_rgb)
            ax.set_title(f'Inference Result — {os.path.basename(TEST_MEDIA_FULL)} | conf≥0.50',
                         fontsize=13, fontweight='bold')
            ax.axis('off')
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, 'inline_preview.png'), dpi=120, bbox_inches='tight')
            plt.show()
        else:
            print(f'Output image not found at: {output_path}')
            print('Check the runs/ folder manually.')

print('\nInference test complete.')

## 📦 STEP 11 — Export Best Model for Deployment

In [ ]:
# Export to ONNX format for deployment on the edge device
print('Exporting best model to ONNX...')

export_model = YOLO(BEST_MODEL_PATH)
export_path = export_model.export(
    format='onnx',
    imgsz=640,
    dynamic=False,     # static shape — faster on fixed-resolution cameras
    simplify=True,     # simplify ONNX graph for faster inference
    opset=12
)

print(f'ONNX model exported to: {export_path}')

# Also keep the PyTorch best.pt for future fine-tuning
print(f'PyTorch model at     : {BEST_MODEL_PATH}')

print('\n' + '=' * 55)
print('TRAINING PIPELINE COMPLETE')
print('=' * 55)
print('Files to copy to your deployment device:')
print(f'  best.pt  → {BEST_MODEL_PATH}')
print(f'  best.onnx→ {str(export_path)}')
print('\nTo run inference in your Home Guardian backend:')
print('  from ultralytics import YOLO')
print(f'  model = YOLO("{BEST_MODEL_PATH}")')
print('  results = model.predict(frame, conf=0.50, iou=0.6)')